# Automated Clickbait Headline Generator — Demo

This notebook demonstrates the end-to-end pipeline:
1. Load a trained Seq2Seq (Encoder-Decoder) checkpoint
2. Feed an article and generate clickbait headlines for it
3. Explore the effect of sampling temperature
4. Visualise the training loss / perplexity curves


In [ ]:
import sys
import os

sys.path.insert(0, os.path.join('..', 'src'))

import json
import math
import torch
import matplotlib.pyplot as plt

from generate import load_model, generate_headline, generate_batch

CHECKPOINT = '../checkpoints/best_model.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Trained Model

In [ ]:
model, word2idx, idx2word, max_article_len = load_model(CHECKPOINT, device)
print(f'Vocabulary size:  {len(word2idx)}')
print(f'Max article len:  {max_article_len}')
print(f'Model:            {type(model).__name__}')

## 2. Generate Headlines from an Article

In [ ]:
article = (
    "A new study published in the journal Nature found that eating chocolate every day "
    "may significantly improve memory and cognitive function in older adults. "
    "Researchers at the University of California tracked 1,000 participants over five years "
    "and found a strong correlation between dark chocolate consumption and reduced risk "
    "of Alzheimer's disease. The lead scientist called the results 'surprising but consistent'."
)

print(f'Article:\n{article}\n')
print('=== Generated Headlines (temperature=1.0) ===')
for i, h in enumerate(
    generate_batch(model, word2idx, idx2word, article_text=article,
                   num_headlines=5, max_article_len=max_article_len,
                   temperature=1.0, device=device),
    1,
):
    print(f'  {i}. {h}')

## 3. Different Articles → Different Headlines

In [ ]:
articles = [
    (
        "Scientists have discovered a planet outside our solar system that may have liquid "
        "water on its surface, raising hopes for extraterrestrial life. The planet, named "
        "Kepler-452b, orbits a sun-like star and sits in the habitable zone."
    ),
    (
        "The stock market plunged over 800 points on Monday after new inflation data showed "
        "consumer prices rising faster than expected. Investors fear the Federal Reserve will "
        "raise interest rates more aggressively than previously anticipated."
    ),
    (
        "A local high school student invented a low-cost water purification device using "
        "recycled materials. The device can produce clean drinking water from contaminated "
        "sources and won her first place at the national science fair."
    ),
]

for art in articles:
    print(f'Article: {art[:100]}...\n')
    headlines = generate_batch(model, word2idx, idx2word,
                               article_text=art, num_headlines=3,
                               max_article_len=max_article_len,
                               temperature=0.8, device=device)
    for h in headlines:
        print(f'  • {h}')
    print()

## 4. Effect of Sampling Temperature

In [ ]:
article = (
    "Researchers at MIT have developed a new artificial intelligence system that can "
    "diagnose rare diseases with higher accuracy than experienced doctors. The system "
    "was trained on millions of medical records and can identify patterns that humans miss."
)

print(f'Article: {article[:120]}...\n')
print('Low T  = more deterministic / repetitive')
print('High T = more creative / random\n')
for temp in [0.4, 0.7, 1.0, 1.3]:
    h = generate_headline(model, word2idx, idx2word,
                          article_text=article,
                          max_article_len=max_article_len,
                          temperature=temp, device=device)
    print(f'  T={temp:.1f}  →  {h}')

## 5. Training Loss & Perplexity Curves

In [ ]:
history_path = '../checkpoints/history.json'

with open(history_path) as f:
    history = json.load(f)

train_losses = history['train_loss']
val_losses   = history['val_loss']
epochs = range(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, label='Train Loss')
axes[0].plot(epochs, val_losses,   label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

train_ppl = [math.exp(l) for l in train_losses]
val_ppl   = [math.exp(l) for l in val_losses]
axes[1].plot(epochs, train_ppl, label='Train Perplexity')
axes[1].plot(epochs, val_ppl,   label='Val Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Training and Validation Perplexity')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()